In [26]:
from langchain.tools import tool, InjectedToolArg
from dotenv import load_dotenv
from langchain.messages import HumanMessage
import os
import requests
import json
from typing import Annotated

In [5]:
load_dotenv()

EXCHANGE_API = os.getenv('exchange_api')

In [7]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between given base currency and target currency
    """
    url = f'https://v6.exchangerate-api.com/v6/{EXCHANGE_API}/pair/{base_currency}/{target_currency}'

    response = requests.get(url)

    return response.json()

In [13]:
@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """given a currency conversion rate this function calculates the target currency value"""
    return base_currency_value*conversion_rate

In [8]:
get_conversion_factor.invoke({'base_currency': 'INR', 'target_currency': 'USD'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1766880001,
 'time_last_update_utc': 'Sun, 28 Dec 2025 00:00:01 +0000',
 'time_next_update_unix': 1766966401,
 'time_next_update_utc': 'Mon, 29 Dec 2025 00:00:01 +0000',
 'base_code': 'INR',
 'target_code': 'USD',
 'conversion_rate': 0.01112}

In [10]:
convert.invoke({'base_currency_value': 10, 'conversion_rate': 85})

850.0

In [15]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')

model = ChatGroq(
    model="llama-3.1-8b-instant"
)

In [16]:
model_with_tools = model.bind_tools([get_conversion_factor, convert])

In [35]:
messages = []

query = 'what is conversion rate from USD to INR, what will be value of 10 USD into INR, what is your source of knowledge, when it got updated?'

human_message = HumanMessage(query)

messages.append(human_message)

response = model_with_tools.invoke(messages)

messages.append(response)

In [36]:
for tool_call in response.tool_calls:
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        messages.append(tool_message1)
    if tool_call['name'] == 'convert':
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

In [37]:
messages

[HumanMessage(content='what is conversion rate from USD to INR, what will be value of 10 USD into INR, what is your source of knowledge, when it got updated?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '7mye74qjq', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'tdqyc63ag', 'function': {'arguments': '{"base_currency":"USD","base_currency_value":10,"target_currency":"INR"}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 177, 'prompt_tokens': 343, 'total_tokens': 520, 'completion_time': 0.291898258, 'completion_tokens_details': None, 'prompt_time': 0.022535389, 'prompt_tokens_details': None, 'queue_time': 0.04999105, 'total_time': 0.314433647}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 

In [38]:
model_with_tools.invoke(messages)

AIMessage(content='The conversion rate from USD to INR is 89.947, and the value of 10 USD is 89.947 INR. My source of knowledge is Exchangerate-api, and it was updated on Sunday, 28th December 2025 at 00:00:01 UTC.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 558, 'total_tokens': 621, 'completion_time': 0.130236371, 'completion_tokens_details': None, 'prompt_time': 0.043763446, 'prompt_tokens_details': None, 'queue_time': 0.059888774, 'total_time': 0.173999817}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b6474-da71-7031-9928-6bb46c4014bf-0', usage_metadata={'input_tokens': 558, 'output_tokens': 63, 'total_tokens': 621})